# Gold dimensions: `dim_region` and `dim_region_boundary`

## Purpose

The official region list that every regional fact joins to, plus each region's shape for maps.

## Grain

- `dim_region`: one official PSGC region per PSGC version, plus key `0`. Uniqueness: `psgc_region_code`, `psgc_version`.
- `dim_region_boundary`: one boundary version per official region. Uniqueness: `region_key`, `boundary_version`.

## Keys

- `region_key`: `XXHASH64` of `PSGC_REGION`, the PSGC code, and the PSGC version. Natural key: `psgc_region_code`, `psgc_version`.
- `boundary_key`: `XXHASH64` of `REGION_BOUNDARY`, the region key, and the boundary version. Natural key: `region_key`, `boundary_version`.

## Key 0 member

`region_key = 0`, "Unknown or non-geographic". Facts use it for Central Office, unmatched, and ambiguous projects. Population ratios exclude it. `dim_region_boundary` has no key 0.

## Inputs

- `02-silver.silver_psgc_place` (region rows)
- `02-silver.silver_region_flood_exposure` (Table 14 boundary status)
- `01-bronze.boundaries` (region shapes, allowed by D-35)
- `04-validation.silver_dq_results`

## Outputs

`03-gold.dim_region` and `03-gold.dim_region_boundary`, loaded with `MERGE`. Rows are never deleted.

## Not done here

No spatial intersection, clipping, or area work. Boundary safety is decided by Table 14 in Silver. Ambiguous shapes are skipped and counted, not repaired.

## 1. Confirm upstream safety and select versions

**What:** declare the rule versions and the BARMM code once. Then check that the PSGC region set and Table 14 are each from one run and passed their Silver validation.

**Why:** Gold must not publish keys from an unsafe or mixed Silver state. Table 14 must also come from the same PSGC run, or its boundary statuses would describe a different region list.

**Protects:** every check uses `ASSERT_TRUE`, so the notebook stops before it writes anything.

In [ ]:
%sql
USE CATALOG `buildabida-capstone`;
CREATE SCHEMA IF NOT EXISTS `03-gold`;

-- psgc region code of bangsamoro (barmm) in the official psa classification
DECLARE OR REPLACE VARIABLE barmm_psgc_region_code STRING DEFAULT '1900000000';
-- gold rule label. it changes the gold run id when this notebook's logic changes (d-29, d-33)
DECLARE OR REPLACE VARIABLE region_rule_version STRING DEFAULT 'gold-dim-region-v1';
-- gold rule label. it changes the gold run id when this notebook's logic changes (d-29, d-33)
DECLARE OR REPLACE VARIABLE boundary_rule_version STRING DEFAULT 'gold-dim-region-boundary-v1';
-- crs of the parsed region shapes. the table 13 parser returns epsg:4326 (d-35)
DECLARE OR REPLACE VARIABLE boundary_crs STRING DEFAULT 'EPSG:4326';

SELECT ASSERT_TRUE(
    COUNT(*) > 0
    AND COUNT(DISTINCT run_id) = 1
    AND COUNT(DISTINCT source_snapshot_id) = 1
    AND COUNT(DISTINCT psgc_version) <= 1
    AND COUNT(*) = COUNT(DISTINCT psgc_code),
    'STOP: selected official PSGC region set is empty or unsafe'
)
FROM `02-silver`.silver_psgc_place
WHERE place_type = 'REGION';

SELECT ASSERT_TRUE(
    COUNT(*) > 0
    AND COUNT_IF(dq.action = 'stop' AND dq.status IN ('FAIL', 'ERROR')) = 0,
    'STOP: selected Silver PSGC validation is missing or unsafe'
)
FROM `04-validation`.silver_dq_results AS dq
WHERE dq.table_name = 'silver_psgc_place'
  AND dq.pipeline_run_id = (
      SELECT MAX(place.run_id)
      FROM `02-silver`.silver_psgc_place AS place
      WHERE place.place_type = 'REGION'
  );

SELECT ASSERT_TRUE(
    COUNT(*) > 0
    AND COUNT(DISTINCT run_id) = 1
    AND COUNT(DISTINCT boundary_source_snapshot_id) = 1,
    'STOP: silver_region_flood_exposure must be nonempty and from one run'
)
FROM `02-silver`.silver_region_flood_exposure;

SELECT ASSERT_TRUE(
    COUNT(*) > 0
    AND COUNT_IF(dq.action = 'stop' AND dq.status IN ('FAIL', 'ERROR')) = 0,
    'STOP: selected Silver regional flood-exposure validation is missing or unsafe'
)
FROM `04-validation`.silver_dq_results AS dq
WHERE dq.table_name = 'silver_region_flood_exposure'
  AND dq.pipeline_run_id = (
      SELECT MAX(exposure.run_id) FROM `02-silver`.silver_region_flood_exposure AS exposure
  );

CREATE OR REPLACE TEMPORARY VIEW gold_region_versions AS
WITH psgc_context AS (
    SELECT
        MAX(run_id) AS psgc_run_id,
        MAX(source_snapshot_id) AS psgc_source_snapshot_id,
        MAX(psgc_version) AS psgc_version,
        MAX(source_load_ts) AS psgc_source_load_ts
    FROM `02-silver`.silver_psgc_place
    WHERE place_type = 'REGION'
),
exposure_context AS (
    SELECT
        MAX(run_id) AS exposure_run_id,
        MAX(psgc_run_id) AS exposure_psgc_run_id,
        MAX(boundary_source_snapshot_id) AS boundary_source_snapshot_id
    FROM `02-silver`.silver_region_flood_exposure
)
SELECT
    psgc_context.psgc_run_id,
    psgc_context.psgc_source_snapshot_id,
    psgc_context.psgc_version,
    psgc_context.psgc_source_load_ts,
    exposure_context.exposure_run_id,
    exposure_context.exposure_psgc_run_id,
    exposure_context.boundary_source_snapshot_id,
    SHA2(CONCAT_WS(
        '|', psgc_context.psgc_run_id, exposure_context.exposure_run_id, region_rule_version
    ), 256) AS region_run_id,
    SHA2(CONCAT_WS(
        '|', psgc_context.psgc_run_id, exposure_context.exposure_run_id,
        exposure_context.boundary_source_snapshot_id, boundary_rule_version
    ), 256) AS boundary_run_id
FROM psgc_context
CROSS JOIN exposure_context;

SELECT ASSERT_TRUE(
    exposure_psgc_run_id <=> psgc_run_id,
    'STOP: silver_region_flood_exposure was built from a different PSGC run than silver_psgc_place'
)
FROM gold_region_versions;

## 2. Read boundary safety and parse region shapes

**What:** take one boundary status per region from Table 14. Then parse the region shapes from the same boundary snapshot Table 14 used.

**Why:** Table 14 already decided which region boundaries are safe. Regions VI, VII, IX, and BARMM have shapes whose territory no longer matches the current PSGC. Negros Island Region has no shape. Gold keeps those findings instead of judging the shapes again. The Bronze read is allowed by D-35. The region filter on `administrative_level` is a small Gold derivation (D-38).

**Protects:** the parser is Table 13's expression. Source coordinates are quoted strings, so the parser removes quotes only around numbers.

The two patterns are raw literals, `r'...'`. The VS Code `%sql` runner turns `\\` into `\`, which breaks an escaped pattern and leaves every shape `NULL`. A raw literal gives the same regex in VS Code and in the workspace. Gold code never uses `\\`.

**Expected validation:** each region has exactly one Table 14 status.

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW region_boundary_status AS
SELECT
    psgc_region_code,
    COUNT(DISTINCT region_boundary_status) AS status_count,
    MAX(region_boundary_status) AS region_boundary_status
FROM `02-silver`.silver_region_flood_exposure
GROUP BY psgc_region_code;

SELECT ASSERT_TRUE(
    COUNT(*) > 0 AND COUNT_IF(status_count <> 1) = 0,
    'STOP: a region has more than one Table 14 boundary status'
)
FROM region_boundary_status;

CREATE OR REPLACE TEMPORARY VIEW gold_region_boundary_parsed AS
SELECT
    boundary.source_feature_id,
    boundary.psgc_code AS psgc_region_code,
    boundary.source_boundary_version AS boundary_version,
    boundary._source_system AS boundary_source_system,
    boundary._ingested_at AS boundary_source_load_ts,
    TRY_TO_GEOMETRY(REGEXP_REPLACE(
        REGEXP_REPLACE(
            boundary.geometry_json,
            r'"(\[[^"]*\])"',
            CONCAT(CHAR(36), '1')
        ),
        r'"(-?[0-9]+(?:\.[0-9]+)?(?:[Ee][+-]?[0-9]+)?)"',
        CONCAT(CHAR(36), '1')
    )) AS boundary_geometry
FROM `01-bronze`.boundaries AS boundary
CROSS JOIN gold_region_versions AS selected
WHERE LOWER(TRIM(boundary.administrative_level)) = 'region'
  AND boundary._source_snapshot_id = selected.boundary_source_snapshot_id;

## 3. Create the tables

**What:** create both tables with the column names, types, and order in `docs/gold_model.md`.

**Why:** facts and the dashboard depend on these exact names. Genie reads every comment.

`boundary_geometry` is `GEOMETRY(4326)`. The parsed shapes carry SRID 4326, so the column states it.

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS `03-gold`.dim_region (
    region_key BIGINT NOT NULL COMMENT 'Surrogate key. XXHASH64 of the label PSGC_REGION, the PSGC region code, and the PSGC version. Key 0 is unknown or non-geographic.',
    psgc_region_code STRING COMMENT 'Official 10-digit PSGC region code. NULL for key 0.',
    region_name STRING COMMENT 'Official PSGC region name, such as Region I (Ilocos Region).',
    region_short_name STRING COMMENT 'Short label from the trailing parentheses of the official name, such as NCR. The full name when there are no parentheses.',
    psgc_version STRING COMMENT 'PSGC publisher version of the official region list.',
    is_barmm BOOLEAN COMMENT 'True for the Bangsamoro Autonomous Region in Muslim Mindanao. Source coverage there is limited.',
    is_geographic_region BOOLEAN COMMENT 'True for an official region. False for key 0, which also holds Central Office.',
    centroid_latitude DOUBLE COMMENT 'Latitude of the boundary centroid in EPSG:4326. NULL when Table 14 found no safe boundary.',
    centroid_longitude DOUBLE COMMENT 'Longitude of the boundary centroid in EPSG:4326. NULL when Table 14 found no safe boundary.',
    run_id STRING COMMENT 'Deterministic Gold run ID. Hash of the Silver PSGC run, the Table 14 run, and the Gold rule version.',
    source_load_ts TIMESTAMP COMMENT 'Bronze load time of the PSGC snapshot behind this row.'
)
USING DELTA
COMMENT 'Gold region dimension. One official PSGC region per PSGC version plus key 0 for unknown or non-geographic records such as Central Office.';

CREATE TABLE IF NOT EXISTS `03-gold`.dim_region_boundary (
    boundary_key BIGINT NOT NULL COMMENT 'Surrogate key. XXHASH64 of the label REGION_BOUNDARY, the region key, and the boundary version.',
    region_key BIGINT NOT NULL COMMENT 'Region this boundary belongs to. Joins to dim_region.region_key.',
    boundary_geometry GEOMETRY(4326) COMMENT 'Region polygon or multipolygon in EPSG:4326, parsed with the Table 13 rule. For maps only, not for area.',
    boundary_version STRING COMMENT 'Publisher version of the boundary dataset.',
    coordinate_reference_system STRING COMMENT 'Coordinate reference system of boundary_geometry. Always EPSG:4326.',
    geometry_valid_flag BOOLEAN COMMENT 'True when the parsed shape is a valid geometry. False when it is invalid or did not parse.',
    psgc_match_status STRING COMMENT 'Boundary safety from Silver Table 14. VALID_REGION_BOUNDARY is safe. BOUNDARY_TERRITORY_MISMATCH means the shape predates the current PSGC territory.',
    source_system STRING COMMENT 'Source system of the boundary snapshot.',
    run_id STRING COMMENT 'Deterministic Gold run ID. Hash of the PSGC run, the Table 14 run, the boundary snapshot, and the Gold rule version.',
    source_load_ts TIMESTAMP COMMENT 'Bronze load time of the boundary snapshot.'
)
USING DELTA
COMMENT 'Gold region boundary dimension. One boundary version per official region. Regions without a boundary feature, such as Negros Island Region, have no row.';

## 4. Build and merge `dim_region`

**What:** build one row per official region, add key `0`, and merge by `region_key`.

**Why:**

- `region_key` is `XXHASH64` of the label `PSGC_REGION`, the region code, and the PSGC version (D-33). The same inputs always give the same key.
- `region_short_name` comes from the trailing parentheses of the official name (D-38).
- `is_barmm` compares the code with the declared BARMM code.
- The centroid comes only from a boundary that Table 14 marked `VALID_REGION_BOUNDARY`. Unsafe or missing shapes leave it `NULL`.

**Expected validation:** keys are unique, and exactly one row has key `0`.

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW dim_region_source AS
WITH official_regions AS (
    SELECT
        place.psgc_code,
        place.place_name_raw,
        place.psgc_version,
        place.source_snapshot_id
    FROM `02-silver`.silver_psgc_place AS place
    WHERE place.place_type = 'REGION'
),
safe_centroids AS (
    SELECT
        parsed.psgc_region_code,
        ST_Y(ST_CENTROID(parsed.boundary_geometry)) AS centroid_latitude,
        ST_X(ST_CENTROID(parsed.boundary_geometry)) AS centroid_longitude
    FROM gold_region_boundary_parsed AS parsed
    INNER JOIN region_boundary_status AS status
        ON parsed.psgc_region_code = status.psgc_region_code
    WHERE status.region_boundary_status = 'VALID_REGION_BOUNDARY'
),
geographic_regions AS (
    SELECT
        XXHASH64(CONCAT_WS(
            '|', 'PSGC_REGION', region.psgc_code,
            COALESCE(region.psgc_version, region.source_snapshot_id)
        )) AS region_key,
        region.psgc_code AS psgc_region_code,
        region.place_name_raw AS region_name,
        COALESCE(
            NULLIF(TRIM(REGEXP_EXTRACT(region.place_name_raw, '[(]([^()]+)[)] *$', 1)), ''),
            region.place_name_raw
        ) AS region_short_name,
        region.psgc_version,
        region.psgc_code = barmm_psgc_region_code AS is_barmm,
        TRUE AS is_geographic_region,
        centroid.centroid_latitude,
        centroid.centroid_longitude,
        selected.region_run_id AS run_id,
        selected.psgc_source_load_ts AS source_load_ts
    FROM official_regions AS region
    CROSS JOIN gold_region_versions AS selected
    LEFT JOIN safe_centroids AS centroid
        ON region.psgc_code = centroid.psgc_region_code
),
unknown_region AS (
    SELECT
        CAST(0 AS BIGINT) AS region_key,
        CAST(NULL AS STRING) AS psgc_region_code,
        'Unknown or non-geographic' AS region_name,
        'Unknown' AS region_short_name,
        selected.psgc_version,
        FALSE AS is_barmm,
        FALSE AS is_geographic_region,
        CAST(NULL AS DOUBLE) AS centroid_latitude,
        CAST(NULL AS DOUBLE) AS centroid_longitude,
        selected.region_run_id AS run_id,
        selected.psgc_source_load_ts AS source_load_ts
    FROM gold_region_versions AS selected
)
SELECT * FROM unknown_region
UNION ALL
SELECT * FROM geographic_regions;

SELECT ASSERT_TRUE(
    COUNT(*) = COUNT(DISTINCT region_key)
    AND COUNT_IF(region_key = 0) = 1
    AND COUNT_IF(region_key = 0 AND is_geographic_region) = 0,
    'STOP: dim_region keys are not unique or collide with key 0'
)
FROM dim_region_source;

MERGE INTO `03-gold`.dim_region AS target
USING dim_region_source AS source
ON target.region_key = source.region_key
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

## 5. Build and merge `dim_region_boundary`

**What:** build one row per region shape whose code is a current official region, and merge by `boundary_key`.

**Why:** the dashboard map needs the shape and its safety status together. Unsafe shapes stay visible with their Table 14 status. They are not hidden and not repaired.

A region that Table 14 marks `AMBIGUOUS_REGION_BOUNDARY` has more than one shape. Gold skips it rather than pick one. Table 14 keeps the finding. Shapes whose code is not a current PSGC region, or that have no Table 14 status, are also skipped. The last query counts each skipped group, so no row disappears silently.

**Expected validation:** one row per `region_key` and `boundary_version`.

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW dim_region_boundary_source AS
SELECT
    XXHASH64(CONCAT_WS(
        '|', 'REGION_BOUNDARY', CAST(region.region_key AS STRING), parsed.boundary_version
    )) AS boundary_key,
    region.region_key,
    ST_SETSRID(parsed.boundary_geometry, 4326) AS boundary_geometry,
    parsed.boundary_version,
    boundary_crs AS coordinate_reference_system,
    COALESCE(ST_ISVALID(parsed.boundary_geometry), FALSE) AS geometry_valid_flag,
    status.region_boundary_status AS psgc_match_status,
    parsed.boundary_source_system AS source_system,
    selected.boundary_run_id AS run_id,
    parsed.boundary_source_load_ts AS source_load_ts
FROM gold_region_boundary_parsed AS parsed
CROSS JOIN gold_region_versions AS selected
INNER JOIN dim_region_source AS region
    ON parsed.psgc_region_code = region.psgc_region_code
INNER JOIN region_boundary_status AS status
    ON parsed.psgc_region_code = status.psgc_region_code
WHERE status.region_boundary_status <> 'AMBIGUOUS_REGION_BOUNDARY';

SELECT ASSERT_TRUE(
    COUNT(*) > 0
    AND COUNT(*) = COUNT(DISTINCT boundary_key)
    AND COUNT(*) = COUNT(DISTINCT NAMED_STRUCT('region_key', region_key, 'boundary_version', boundary_version)),
    'STOP: dim_region_boundary is empty or not unique per region and boundary version'
)
FROM dim_region_boundary_source;

MERGE INTO `03-gold`.dim_region_boundary AS target
USING dim_region_boundary_source AS source
ON target.boundary_key = source.boundary_key
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

SELECT
    COUNT(*) AS parsed_region_shapes,
    COUNT_IF(status.region_boundary_status = 'AMBIGUOUS_REGION_BOUNDARY') AS skipped_ambiguous_boundary,
    COUNT_IF(region.region_key IS NULL) AS skipped_not_a_current_psgc_region,
    COUNT_IF(status.psgc_region_code IS NULL) AS skipped_no_table_14_status,
    COUNT_IF(
        region.region_key IS NOT NULL
        AND status.psgc_region_code IS NOT NULL
        AND status.region_boundary_status <> 'AMBIGUOUS_REGION_BOUNDARY'
    ) AS loaded_boundary_rows
FROM gold_region_boundary_parsed AS parsed
LEFT JOIN dim_region_source AS region
    ON parsed.psgc_region_code = region.psgc_region_code
LEFT JOIN region_boundary_status AS status
    ON parsed.psgc_region_code = status.psgc_region_code;

# Summary

Created or refreshed `03-gold.dim_region` and `03-gold.dim_region_boundary`.

- `dim_region` holds every official PSGC region plus key `0` for unknown or non-geographic records.
- Centroids exist only for regions with a safe Table 14 boundary.
- `dim_region_boundary` keeps every current region shape with its Table 14 safety status.
- No spatial matching, clipping, or area work runs here.
- Coverage: the query below shows each region with its boundary status. The cell 10 count shows parsed, loaded, and skipped shapes.
- Known limits: at the Oct 8 run, Regions VI, VII, NIR, and BARMM have no centroid because Table 14 found no safe boundary. NIR has no boundary feature. BARMM "No Data" is a source gap, not low investment.

Next: `02_gold_dim_flood_susceptibility`.

In [ ]:
%sql
SELECT
    region.region_key,
    region.psgc_region_code,
    region.region_short_name,
    region.is_barmm,
    region.centroid_latitude,
    region.centroid_longitude,
    boundary.psgc_match_status,
    boundary.geometry_valid_flag
FROM `03-gold`.dim_region AS region
LEFT JOIN `03-gold`.dim_region_boundary AS boundary
    ON region.region_key = boundary.region_key
ORDER BY region.psgc_region_code NULLS FIRST;